In [1]:
import pandas as pd
import folium
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt

# Load data
df = pd.read_csv('data/ins-3/all_districts_bhumjaithai_ratio.csv')

# Coordinate mapping from existing project data_loader
TAMBON_COORDS = {
    "ตำบลบ้านไร่": (15.107087, 99.634135),
    "ตำบลคอกควาย": (15.23317, 99.389511),
    "ตำบลหนองจอก": (15.046893, 99.689832),
    "ตำบลเจ้าวัด": (15.149801, 99.445044),
    "ตำบลวังหิน": (15.282827, 99.717694),
    "ตำบลทัพหลวง": (15.050689, 99.600735),
    "ตำบลบ้านบึง": (15.029448, 99.556222),
    "ตำบลหูช้าง": (15.140354, 99.667549),
    "ตำบลเมืองการุ้ง": (15.185668, 99.689832),
    "ตำบลแก่นมะกรูด": (15.159935, 99.290317),
    "ตำบลห้วยแห้ง": (15.16825, 99.556222),
    "ตำบลหนองบ่มกล้วย": (15.090257, 99.756716),
    "ตำบลบ้านใหม่คลองเคียน": (15.230143, 99.664764),
    "ตำบลลานสัก": (15.533594, 99.41172),
    "ตำบลทุ่งนางาม": (15.374756, 99.600735),
    "ตำบลน้ำรอบ": (15.550078, 99.567348),
    "ตำบลประดู่ยืน": (15.442343, 99.645271),
    "ตำบลป่าอ้อ": (15.401764, 99.511733),
    "ตำบลระบำ": (15.537375, 99.322921),
    "ตำบลทุ่งโพ": (15.367956, 99.756716),
    "ตำบลเขากวางทอง": (15.394052, 99.689832),
    "ตำบลเขาบางแกรก": (15.32553, 99.667549),
    "ตำบลห้วยคต": (15.259899, 99.578475),
    "ตำบลสุขฤทัย": (15.28018, 99.645271),
    "ตำบลทองหลาง": (15.3406, 99.4795),
}

# Attach coordinates
df['lat'] = df['tambon'].map(lambda x: TAMBON_COORDS.get(x, (None, None))[0])
df['lon'] = df['tambon'].map(lambda x: TAMBON_COORDS.get(x, (None, None))[1])
df = df.dropna(subset=['lat', 'lon'])

# Setup Map
m = folium.Map(location=[15.3, 99.55], zoom_start=10, tiles='CartoDB Positron')

# Define color map function (Blues)
cmap = plt.get_cmap('Blues')
vmin = df['ratio'].min()
vmax = df['ratio'].max()

def get_color(val):
    if pd.isna(val):
        return '#cccccc'
    # Normalize value between 0.3 and 1.0 to avoid too light colors
    norm_val = 0.3 + 0.7 * ((val - vmin) / (vmax - vmin) if vmax > vmin else 1.0)
    return mcolors.to_hex(cmap(norm_val))

# Add circles
for idx, row in df.iterrows():
    color = get_color(row['ratio'])
    
    # Tooltip content
    html = f"""
    <b>Amphoe:</b> {row['amphoe']}<br>
    <b>Tambon:</b> {row['tambon']}<br>
    <b>Loyalty Ratio:</b> {row['ratio']:.1%}<br>
    <b>Votes:</b> {row['bhumjaithai_votes']} / {row['total_valid_ballots']}
    """
    
    folium.Circle(
        location=[row['lat'], row['lon']],
        radius=3000, # 3km radius to cover tambon area visually
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        weight=1,
        tooltip=folium.Tooltip(html)
    ).add_to(m)

# Save to HTML
m.save('loyalty_map_tambon.html')
display(m)
